In [ ]:
# ============================================================
# 8. Per-Scenario Outcome Analysis (TEST SET)
# ============================================================
# Purpose:
# - Analyse final system decisions at the scenario level
# - Identify which true scenarios are correctly detected or misclassified
# - Provide error localisation for interpretability
#
# NOTE:
# - Uses true scenario markers (evaluation only)
# - Evaluates final hierarchical output, not individual models

# ------------------------------------------------------------
# Create correctness flag
# ------------------------------------------------------------
results_df["Correct"] = (
    (results_df["Final_binary"] == 1) &
    (~results_df["True_marker"].isin(NON_ATTACK_IDS))
) | (
    (results_df["Final_binary"] == 0) &
    (results_df["True_marker"].isin(NON_ATTACK_IDS))
)

# ------------------------------------------------------------
# Per-scenario summary
# ------------------------------------------------------------
scenario_summary = (
    results_df
    .groupby(["True_marker", "True_label_name"])
    .agg(
        Samples=("Correct", "size"),
        Accuracy=("Correct", "mean"),
        Attack_Pred_Rate=("Final_binary", "mean")
    )
    .reset_index()
    .sort_values("True_marker")
)

print("Per-Scenario Outcome Summary (TEST SET)")
display(scenario_summary)

# ------------------------------------------------------------
# Highlight weakest-performing scenarios
# ------------------------------------------------------------
print("\nLowest-Accuracy Scenarios:")
display(
    scenario_summary
    .sort_values("Accuracy")
    .head(8)
)

In [ ]:
# ============================================================
# 9. Error / Misrouting Analysis (End-to-End System)
# ============================================================
# Purpose:
# - Explain where end-to-end errors come from
# - Show how M1 conservative routing impacts non-attack scenarios
# - Quantify misrouting patterns by scenario and by decision path
#
# NOTE:
# - With the new M2 design, non-attacks output FAMILY labels only.
# - Therefore, scenario-level correctness is evaluated ONLY at the binary level
#   (Attack vs Non-Attack). Attack subtype correctness is handled separately.

# ------------------------------------------------------------
# 9.1 Binary correctness flag (system-level)
# ------------------------------------------------------------
true_is_attack = ~results_df["True_marker"].isin(NON_ATTACK_IDS)
pred_is_attack = (results_df["Final_binary"] == 1)

results_df["Binary_correct"] = (true_is_attack == pred_is_attack)

# ------------------------------------------------------------
# 9.2 Overall error rate and split by true class
# ------------------------------------------------------------
overall_error = 1 - results_df["Binary_correct"].mean()

non_attack_mask = results_df["True_marker"].isin(NON_ATTACK_IDS)
attack_mask = ~non_attack_mask

non_attack_error = 1 - results_df.loc[non_attack_mask, "Binary_correct"].mean()
attack_error = 1 - results_df.loc[attack_mask, "Binary_correct"].mean()

print("End-to-End Binary Error Summary (TEST SET)")
print(f" Overall error rate:       {overall_error:.4f}")
print(f" Non-attack error rate:    {non_attack_error:.4f}  (driven by conservative routing)")
print(f" Attack error rate:        {attack_error:.4f}")

# ------------------------------------------------------------
# 9.3 Where do non-attacks get misrouted?
# (i.e., true non-attack predicted as attack)
# ------------------------------------------------------------
non_attack_false_alarms = results_df[non_attack_mask & pred_is_attack].copy()

fa_by_scenario = (
    non_attack_false_alarms
    .groupby(["True_marker", "True_label_name"])
    .size()
    .reset_index(name="False_Alarm_Count")
    .sort_values("False_Alarm_Count", ascending=False)
)

print("\nTop non-attack scenarios misrouted as attacks (False Alarms):")
display(fa_by_scenario.head(10))

# Add rate relative to scenario size
scenario_sizes = results_df.groupby("True_marker").size().rename("Scenario_Samples")
fa_by_scenario = fa_by_scenario.merge(
    scenario_sizes.reset_index(),
    on="True_marker",
    how="left"
)
fa_by_scenario["False_Alarm_Rate"] = fa_by_scenario["False_Alarm_Count"] / fa_by_scenario["Scenario_Samples"]

print("\nFalse alarm rate by non-attack scenario:")
display(
    fa_by_scenario.sort_values("False_Alarm_Rate", ascending=False).head(10)
)

# ------------------------------------------------------------
# 9.4 Which decision path is responsible for errors?
# (This is the key "hierarchy behaviour" explanation)
# ------------------------------------------------------------
error_df = results_df[~results_df["Binary_correct"]].copy()

error_by_decision = (
    error_df["Decision"]
    .value_counts(normalize=True)
    .rename("Fraction_of_All_Errors")
    .reset_index()
    .rename(columns={"index": "Decision"})
)

print("\nError attribution by decision path (fraction of all errors):")
display(error_by_decision)

# ------------------------------------------------------------
# 9.5 Decision-path breakdown for non-attack false alarms
# ------------------------------------------------------------
fa_by_decision = (
    non_attack_false_alarms["Decision"]
    .value_counts(normalize=True)
    .rename("Fraction_of_NonAttack_FalseAlarms")
    .reset_index()
    .rename(columns={"index": "Decision"})
)

print("\nNon-attack false alarms: where they occur in the hierarchy")
display(fa_by_decision)

# ------------------------------------------------------------
# 9.6 Compact thesis table: worst non-attack scenarios + routing behaviour
# ------------------------------------------------------------
fa_thesis_table = (
    non_attack_false_alarms
    .groupby(["True_marker", "True_label_name", "Decision"])
    .size()
    .reset_index(name="Count")
    .merge(scenario_sizes.reset_index(), on="True_marker", how="left")
)

fa_thesis_table["Rate_within_scenario"] = fa_thesis_table["Count"] / fa_thesis_table["Scenario_Samples"]

print("\nThesis-ready table: non-attack misroutes by scenario and path")
display(
    fa_thesis_table
    .sort_values(["Rate_within_scenario", "Count"], ascending=False)
    .head(15)
)

In [ ]:
# ============================================================
# Section 10 - Attack Diagnostic Performance (TEST SET)
# ============================================================
# Purpose:
# - Evaluate fine-grained attack diagnosis (M4–M6)
# - Only for samples that reached ATTACK_CONFIRMED
# - Excludes routing and fallback behaviour by design

from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# Filter: true attacks that reached a specialist leaf
# ------------------------------------------------------------
attack_df = results_df[
    (results_df["True_marker"].isin(DI_IDS + RT_IDS + RSC_IDS)) &
    (results_df["Decision"] == "ATTACK_CONFIRMED")
].copy()

print("Attack-only samples used:", len(attack_df))
print("Unique attack scenarios:", attack_df["True_marker"].nunique())

# ------------------------------------------------------------
# Overall attack diagnostic accuracy
# ------------------------------------------------------------
attack_accuracy = accuracy_score(
    attack_df["True_marker"],
    attack_df["Final_class"]
)

print("\nOverall attack diagnostic accuracy:")
print(f" Accuracy: {attack_accuracy:.4f}")

# ------------------------------------------------------------
# Family-level diagnostic accuracy
# ------------------------------------------------------------
def attack_family(marker):
    if marker in DI_IDS:
        return "Data Injection"
    elif marker in RT_IDS:
        return "Remote Tripping"
    elif marker in RSC_IDS:
        return "Relay Setting Change"
    else:
        return "Unknown"

attack_df["Attack_Family"] = attack_df["True_marker"].apply(attack_family)

# ------------------------------------------------------------
# Family-level diagnostic accuracy (future-proof)
# ------------------------------------------------------------
family_accuracy = (
    attack_df
    .groupby("Attack_Family")[["True_marker", "Final_class"]]
    .apply(lambda g: accuracy_score(g["True_marker"], g["Final_class"]))
    .rename("Accuracy")
    .reset_index()
)

print("\nFamily-level diagnostic accuracy:")
display(family_accuracy)

# ------------------------------------------------------------
# Coverage diagnostic (what fraction of attacks were diagnosed)
# ------------------------------------------------------------
total_true_attacks = results_df["True_marker"].isin(DI_IDS + RT_IDS + RSC_IDS).sum()
diagnosed_attacks = len(attack_df)

print("\nDiagnostic coverage:")
print(f" Fraction of true attacks reaching leaf classifiers: "
      f"{diagnosed_attacks / total_true_attacks:.4f}")

In [ ]:

# ============================================================
# Non-Attack Diagnostic (TEST SET)
# Scenario-group aligned, SLG/L2-safe
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Scenario Groups (from your definition)
# ------------------------------------------------------------

SLG_IDS     = [1, 2, 3, 4, 5, 6]
MAINT_IDS   = [13, 14]
NORMAL_IDS  = [41]

NON_ATTACK_IDS = SLG_IDS + MAINT_IDS + NORMAL_IDS

# ------------------------------------------------------------
# Non-Attack Scenario → Family mapping
# ------------------------------------------------------------

NON_ATTACK_FAMILY_LOOKUP = {}

for k in SLG_IDS:
    NON_ATTACK_FAMILY_LOOKUP[k] = "SLG Faults"

for k in MAINT_IDS:
    NON_ATTACK_FAMILY_LOOKUP[k] = "Line Maintenance"

for k in NORMAL_IDS:
    NON_ATTACK_FAMILY_LOOKUP[k] = "Normal Operation"

# ------------------------------------------------------------
# Filter ALL non-attack samples
# ------------------------------------------------------------

non_attack_df = results_df[
    results_df["True_marker"].isin(NON_ATTACK_IDS)
].copy()

# Explicit family column (critical)
non_attack_df["Non_Attack_Family"] = non_attack_df["True_marker"].map(
    NON_ATTACK_FAMILY_LOOKUP
)

# ------------------------------------------------------------
# Aggregate totals and false alarms
# ------------------------------------------------------------

total_counts = (
    non_attack_df
    .groupby("Non_Attack_Family")
    .size()
    .rename("Total_Samples")
)

false_alarm_counts = (
    non_attack_df[non_attack_df["Final_binary"] == 1]
    .groupby("Non_Attack_Family")
    .size()
    .rename("False_Alarms")
)

# ------------------------------------------------------------
# Combine + compute false-alarm rate
# ------------------------------------------------------------

diagnostic_df = (
    pd.concat([total_counts, false_alarm_counts], axis=1)
    .fillna(0)
)

diagnostic_df["False_Alarm_Rate (%)"] = (
    diagnostic_df["False_Alarms"] / diagnostic_df["Total_Samples"]
) * 100

diagnostic_df = diagnostic_df.reset_index()

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("Non-Attack Diagnostic Summary (TEST SET)")
display(diagnostic_df)

In [ ]:

# ============================================================
# Section 11.1 — Non-Attack → Attack Misrouting (FINAL)
# Table + Simple Stacked Bar (Counts Only)
# ============================================================

# ============================================================
# Section 11.1 — Non-Attack False Alarms & Misrouting (ONE TABLE)
# Robust version: NO groupby.apply
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Scenario Groups
# ------------------------------------------------------------
SLG_IDS     = [1, 2, 3, 4, 5, 6]
MAINT_IDS   = [13, 14]
NORMAL_IDS  = [41]

NON_ATTACK_IDS = SLG_IDS + MAINT_IDS + NORMAL_IDS

DI_IDS  = [7, 8, 9, 10, 11, 12]
RT_IDS  = [15, 16, 17, 18, 19, 20]
RSC_IDS = [21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 35, 36, 37, 38, 39, 40]

# ------------------------------------------------------------
# Lookups
# ------------------------------------------------------------
NON_ATTACK_FAMILY_LOOKUP = {}
for k in SLG_IDS:    NON_ATTACK_FAMILY_LOOKUP[k] = "SLG Faults"
for k in MAINT_IDS:  NON_ATTACK_FAMILY_LOOKUP[k] = "Line Maintenance"
for k in NORMAL_IDS: NON_ATTACK_FAMILY_LOOKUP[k] = "Normal Operation"

ATTACK_FAMILY_LOOKUP = {}
for k in DI_IDS:  ATTACK_FAMILY_LOOKUP[k]  = "Data Injection"
for k in RT_IDS:  ATTACK_FAMILY_LOOKUP[k]  = "Remote Tripping"
for k in RSC_IDS: ATTACK_FAMILY_LOOKUP[k]  = "Relay Setting Change"

# ------------------------------------------------------------
# Filter non-attack samples + map families
# ------------------------------------------------------------
df = results_df.loc[
    results_df["True_marker"].isin(NON_ATTACK_IDS),
    ["True_marker", "Final_binary", "Final_class"]
].copy()

df["Non_Attack_Family"] = df["True_marker"].map(NON_ATTACK_FAMILY_LOOKUP)

# Predicted attack family ONLY for false alarms
df["Pred_Attack_Family"] = np.where(
    df["Final_binary"] == 1,
    df["Final_class"].map(ATTACK_FAMILY_LOOKUP),
    np.nan
)

# ------------------------------------------------------------
# Base aggregation (counts)
# ------------------------------------------------------------
base = (
    df.groupby("Non_Attack_Family")
      .agg(
          Total_Samples=("Final_binary", "size"),
          False_Alarms=("Final_binary", "sum")
      )
)

base["False Alarm Rate (%)"] = (
    base["False_Alarms"] / base["Total_Samples"] * 100
)

# ------------------------------------------------------------
# Misrouting counts (false alarms only)
# ------------------------------------------------------------
misroute_counts = (
    df[df["Final_binary"] == 1]
    .groupby(["Non_Attack_Family", "Pred_Attack_Family"])
    .size()
    .unstack(fill_value=0)
)

# Ensure all attack families exist as columns
for col in ["Data Injection", "Relay Setting Change", "Remote Tripping"]:
    if col not in misroute_counts.columns:
        misroute_counts[col] = 0

# ------------------------------------------------------------
# Convert misrouting to percentages
# ------------------------------------------------------------
misroute_pct = (
    misroute_counts
    .div(misroute_counts.sum(axis=1).replace(0, 1), axis=0)
    * 100
)

# ------------------------------------------------------------
# Build FINAL table (one object, no join confusion)
# ------------------------------------------------------------
section_11_1_table = pd.concat(
    [
        base,
        misroute_pct[[
            "Data Injection",
            "Relay Setting Change",
            "Remote Tripping"
        ]].add_suffix(" Misroute (%)")
    ],
    axis=1
).round(1)

# Force preferred order
order = ["SLG Faults", "Line Maintenance", "Normal Operation"]
section_11_1_table = section_11_1_table.reindex(order)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------
print("Table 11.1 — Non-Attack False Alarms and Misrouting Summary (TEST SET)")
display(section_11_1_table)

# ------------------------------------------------------------
# Extract misrouting counts from the same pipeline
# ------------------------------------------------------------

bar_data = misroute_counts.reindex(
    ["SLG Faults", "Line Maintenance", "Normal Operation"],
    fill_value=0
)

# ------------------------------------------------------------
# Plot: stacked horizontal bar (COUNTS)
# ------------------------------------------------------------
plt.figure(figsize=(10, 5))

misrouting_counts.plot(
    kind="barh",
    stacked=True,
    figsize=(10, 5)
)

plt.xlabel("Number of false-alarm samples", fontsize=12)
plt.ylabel("True non-attack family", fontsize=12)
plt.title(
    "Non-Attack → Attack Misrouting (Counts)",
    fontsize=14
)

plt.legend(
    title="Predicted Attack Family",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Section 11.2 — Non-Attack → Attack Misrouting Heatmap
# ============================================================

import seaborn as sns
import matplotlib.pyplot as plt

# ----------------------------
# Pivot table for heatmap
# ----------------------------
heatmap_data = (
    false_alarm_df
    .groupby(["True_label_name", "Final_label"])
    .size()
    .unstack(fill_value=0)
)

# ----------------------------
# Plot heatmap (large, readable)
# ----------------------------
plt.figure(figsize=(16, 8))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt="d",
    cmap="Reds",
    cbar_kws={"label": "Number of samples"}
)

plt.xlabel("Predicted Attack Label", fontsize=12)
plt.ylabel("True Non-Attack Scenario", fontsize=12)
plt.title(
    "Non-Attack → Attack Misrouting Heatmap\n"
    "(False Alarms by Scenario and Predicted Attack)",
    fontsize=14
)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()